In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient


user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("api_key")

# 'GPU T4 x2' or 'GPU P100' in kaggle
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

print(f"Loading model on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=hf_token)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, token=hf_token)
model.to(device)
model.eval() 

tqdm.pandas() # progress bar

def get_multilingual_sentiment(text):
    """Calculates sentiment score (-1 to 1) using XLM-RoBERTa."""
    if not text or len(str(text)) < 5:
        return 0
    
    try:
    
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        
        with torch.no_grad():
            output = model(**encoded_input)
        
        # scores
        scores = output[0][0].detach().cpu().numpy()
        scores = softmax(scores)
        
        # XLM-T output index mapping: 0: Negative, 1: Neutral, 2: Positive
        # Compound score calculation
        ranking = scores[2] - scores[0] 
        return ranking
    except Exception as e:
        return 0

def process_file(file_path, output_name):
    """Reads CSV from Kaggle input, cleans/scores, and saves to Kaggle working dir."""
    target_folder = "/kaggle/working/"
    os.makedirs(target_folder, exist_ok=True)
    
    print(f"\nProcessing: {file_path}")
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    df = pd.read_csv(file_path, lineterminator='\n')
    
    print(f"Analyzing sentiment for {len(df)} rows...")
    df['sentiment_score'] = df['tweet'].progress_apply(get_multilingual_sentiment)
    
    output_path = os.path.join(target_folder, output_name)
    df.to_csv(output_path, index=False)
    print(f"Done! Saved to {output_path}")


trump_input = "/kaggle/input/datasets/mckaylana/trump-and-biden/trump_preprocessed.csv"
biden_input = "/kaggle/input/datasets/mckaylana/trump-and-biden/biden_preprocessed.csv"

process_file(trump_input, 'trump_translated.csv')
process_file(biden_input, 'biden_translated.csv')

Loading model on cuda...


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...


Processing: /kaggle/input/datasets/mckaylana/trump-and-biden/trump_preprocessed.csv
Analyzing sentiment for 320620 rows...


  0%|          | 0/320620 [00:00<?, ?it/s]

Done! Saved to /kaggle/working/trump_translated.csv

Processing: /kaggle/input/datasets/mckaylana/trump-and-biden/biden_preprocessed.csv
Analyzing sentiment for 260195 rows...


  0%|          | 0/260195 [00:00<?, ?it/s]

Done! Saved to /kaggle/working/biden_translated.csv
